In [ ]:
# ---------------- Imports ----------------
import json
import os
import sys

import yaml

from transformers import AutoTokenizer



In [ ]:
# ---------------- Config ----------------
dataset_folder = "yield-v1-finetuning/train"



In [ ]:
# ---------------- Config ----------------
with open("../../config/config.yaml", "r") as f:
    config = yaml.safe_load(f)

data_path = os.path.join(config["paths"]["proj_store"], "data")
models_folderpath = config["paths"]["models"]



In [ ]:
# Load LLaMA tokenizer
MODEL_NAME = f"{models_folderpath}/meta-llama/Meta-Llama-3-8B-Instruct" 
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)

def count_tokens_in_jsonl(jsonl_file):
    token_counts = []
    with open(jsonl_file, 'r', encoding='utf-8') as f:
        for idx, line in enumerate(f):
            try:
                obj = json.loads(line)
                messages = obj.get("messages", [])
                full_text = "\n".join([msg["content"] for msg in messages])
                tokens = tokenizer(full_text, return_tensors=None, add_special_tokens=False)["input_ids"]
                token_count = len(tokens)
                token_counts.append(token_count)
                #print(f"{os.path.basename(jsonl_file)} - Line {idx+1}: {token_count} tokens")
            except Exception as e:
                #print(f"{os.path.basename(jsonl_file)} - Error parsing line {idx+1}: {e}")
                continue
    
    return token_counts

def count_tokens_in_folder(folder_path):
    all_token_counts = []
    jsonl_files = [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith('.jsonl')]

    if not jsonl_files:
        print(f"No .jsonl files found in {folder_path}")
        return

    for jsonl_file in jsonl_files:
        print(f"\nProcessing file: {jsonl_file}")
        file_token_counts = count_tokens_in_jsonl(jsonl_file)

        if file_token_counts:
            print(f"\nSummary for {os.path.basename(jsonl_file)}:")
            print(f"Total examples: {len(file_token_counts)}")
            print(f"Min tokens: {min(file_token_counts)}")
            print(f"Max tokens: {max(file_token_counts)}")
            print(f"Average tokens: {sum(file_token_counts)/len(file_token_counts):.2f}")

        all_token_counts.extend(file_token_counts)

    if all_token_counts:
        print("\n=== Overall Summary ===")
        print(f"Total examples: {len(all_token_counts)}")
        print(f"Min tokens: {min(all_token_counts)}")
        print(f"Max tokens: {max(all_token_counts)}")
        print(f"Average tokens: {sum(all_token_counts)/len(all_token_counts):.2f}")
    else:
        print("No valid lines processed in any file.")



In [ ]:
# Use
jsonl_folder = os.path.join(data_path, dataset_folder)
count_tokens_in_folder(jsonl_folder)

